<a href="https://colab.research.google.com/github/SAMBOAH95/LHBAP-IoT-Mutual-Authentication/blob/thesis-final-implementation/LHBAP_Complete_Comparative_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LHBAP: Integrated Computational and Communication Cost Evaluation

**Purpose:** Reproduce the proposed LHBAP simulation and calculate the comparative computational and communication costs using the equations, operation counts, and message-field sizes already stated in the submitted thesis/manuscript.

**Important methodological distinction:** LHBAP is executed as a complete software simulation. The comparison protocols are evaluated using the operation-count equations documented in the thesis and measured cryptographic primitive costs. This notebook does **not** claim that the complete published implementations of the comparison protocols have been reconstructed.

The equations are retained:
- Proposed LHBAP: \(6T_{HMAC}\)
- Mostafa et al. (2020): \(8T_{HMAC}+2T_{AES}+2T_{PUF}\)
- Wang et al. (2025) / EBIAS: \(6T_{EC}+8T_H\)
- He et al. (2023): \(10T_{EC}+13T_H\)

The Auth1, Auth2, and session-key equations used by LHBAP are not changed in this notebook.

## 1. Imports and reproducibility settings

Run the notebook from top to bottom in a fresh Google Colab runtime. The benchmark uses warm-up runs, repeated batches, and reports the mean and standard deviation. Cloud-runtime measurements can vary because Colab resources are shared.

In [ ]:
import sys, os, platform, time, hmac, hashlib, secrets, statistics, getpass
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import ecdsa
except ImportError:
    %pip -q install ecdsa
    import ecdsa

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("Logical CPU cores:", os.cpu_count())
try:
    import psutil
    print("Available RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 2))
except Exception:
    print("RAM information: psutil unavailable")
print("ecdsa version:", getattr(ecdsa, "__version__", "unknown"))
print("Benchmark timestamp (UTC):", datetime.now(timezone.utc).isoformat())
print("Timer: time.perf_counter_ns()")

## 2. LHBAP protocol equations retained from the manuscript

The implementation below follows the existing equations:

\[
K_{sg}=HMAC(K_{master}, ID_{SN}\parallel ID_{GW})
\]

\[
Auth_1=HMAC(K_{sg}, ID_{SN}\parallel N_s\parallel T_s)
\]

\[
Auth_2=HMAC(K_{sg}, ID_{GW}\parallel N_g\parallel T_g)
\]

\[
SK=HMAC(K_{sg}, ID_{SN}\parallel ID_{GW}\parallel N_s\parallel N_g\parallel T_s\parallel T_g)
\]

The implementation uses timestamp freshness validation and records previously accepted nonces to test duplicate-message rejection. The nonce records are in-memory simulation state; production deployment would require secure, appropriately persistent replay-state management.

**Benchmark boundary:** the pairwise key is derived once during the secure-provisioning stage and supplied to both entities. That provisioning operation is excluded from the online mutual-authentication/session-key execution benchmark, consistent with the thesis's six-HMAC online operation count.

In [ ]:
# Protocol configuration
MASTER_KEY = secrets.token_bytes(32)
DELTA_T = 30  # seconds; retain the value used in the thesis
ID_SN = "SN-001"
ID_GW = "GW-001"

def _b(value):
    return str(value).encode("utf-8")

def hmac_sha256(key: bytes, message: bytes) -> bytes:
    return hmac.new(key, message, hashlib.sha256).digest()

def derive_pairwise_key(master_key: bytes, id_sn: str, id_gw: str) -> bytes:
    return hmac_sha256(master_key, _b(id_sn) + b"||" + _b(id_gw))

# Secure provisioning is modelled once, outside the online authentication benchmark.
PROVISIONED_K_SG = derive_pairwise_key(MASTER_KEY, ID_SN, ID_GW)

class Gateway:
    def __init__(self, k_sg, id_gw, delta_t=30):
        self.id_gw = id_gw
        self.delta_t = delta_t
        self.used_sensor_nonces = set()
        self.k_sg = k_sg

    def verify_sensor(self, message):
        id_sn, ns, ts, auth1 = message
        if abs(int(time.time()) - int(ts)) > self.delta_t:
            return False
        expected = hmac_sha256(
            self.k_sg, _b(id_sn) + b"||" + _b(ns) + b"||" + _b(ts)
        )
        if not hmac.compare_digest(expected, auth1):
            return False
        if ns in self.used_sensor_nonces:
            return False
        self.used_sensor_nonces.add(ns)
        return True

    def create_response(self):
        ng = secrets.token_hex(16)
        tg = int(time.time())
        auth2 = hmac_sha256(
            self.k_sg, _b(self.id_gw) + b"||" + _b(ng) + b"||" + _b(tg)
        )
        return (self.id_gw, ng, tg, auth2)

    def derive_session_key(self, id_sn, ns, ts, ng, tg):
        data = ( _b(id_sn) + b"||" + _b(self.id_gw) + b"||" +
                 _b(ns) + b"||" + _b(ng) + b"||" + _b(ts) + b"||" + _b(tg) )
        return hmac_sha256(self.k_sg, data)

class SensorNode:
    def __init__(self, k_sg, id_sn, id_gw, delta_t=30):
        self.id_sn = id_sn
        self.id_gw = id_gw
        self.delta_t = delta_t
        self.used_gateway_nonces = set()
        self.k_sg = k_sg

    def create_request(self):
        ns = secrets.token_hex(16)
        ts = int(time.time())
        auth1 = hmac_sha256(
            self.k_sg, _b(self.id_sn) + b"||" + _b(ns) + b"||" + _b(ts)
        )
        return (self.id_sn, ns, ts, auth1)

    def verify_gateway(self, message):
        id_gw, ng, tg, auth2 = message
        if abs(int(time.time()) - int(tg)) > self.delta_t:
            return False
        expected = hmac_sha256(
            self.k_sg, _b(id_gw) + b"||" + _b(ng) + b"||" + _b(tg)
        )
        if not hmac.compare_digest(expected, auth2):
            return False
        if ng in self.used_gateway_nonces:
            return False
        self.used_gateway_nonces.add(ng)
        return True

    def derive_session_key(self, ns, ts, ng, tg):
        data = ( _b(self.id_sn) + b"||" + _b(self.id_gw) + b"||" +
                 _b(ns) + b"||" + _b(ng) + b"||" + _b(ts) + b"||" + _b(tg) )
        return hmac_sha256(self.k_sg, data)

def run_complete_protocol():
    sensor = SensorNode(PROVISIONED_K_SG, ID_SN, ID_GW, DELTA_T)
    gateway = Gateway(PROVISIONED_K_SG, ID_GW, DELTA_T)
    m1 = sensor.create_request()
    if not gateway.verify_sensor(m1):
        return False
    m2 = gateway.create_response()
    if not sensor.verify_gateway(m2):
        return False
    sk_sensor = sensor.derive_session_key(m1[1], m1[2], m2[1], m2[2])
    sk_gateway = gateway.derive_session_key(m1[0], m1[1], m1[2], m2[1], m2[2])
    return hmac.compare_digest(sk_sensor, sk_gateway)

## 3. Functional and replay-detection tests

These tests verify the implementation behaviour in this notebook. They are simulation tests, not automated formal verification.

In [ ]:
def test_protocol_and_replay():
    sensor = SensorNode(PROVISIONED_K_SG, ID_SN, ID_GW, DELTA_T)
    gateway = Gateway(PROVISIONED_K_SG, ID_GW, DELTA_T)

    m1 = sensor.create_request()
    legitimate_m1 = gateway.verify_sensor(m1)
    replayed_m1 = gateway.verify_sensor(m1)

    m2 = gateway.create_response()
    legitimate_m2 = sensor.verify_gateway(m2)
    replayed_m2 = sensor.verify_gateway(m2)

    return {
        "Legitimate sensor request accepted": legitimate_m1,
        "Repeated sensor request rejected": not replayed_m1,
        "Legitimate gateway response accepted": legitimate_m2,
        "Repeated gateway response rejected": not replayed_m2,
        "Complete protocol successful": run_complete_protocol()
    }

test_results = test_protocol_and_replay()
display(pd.DataFrame([{"Test": k, "Result": "PASS" if v else "FAIL"} for k,v in test_results.items()]))
assert all(test_results.values()), "At least one functional/replay test failed."

## 4. Complete LHBAP execution-time benchmark

This times the complete simulated LHBAP execution, including object creation, nonce generation, timestamp handling, HMAC computations, verification, and session-key derivation. It is not directly interchangeable with the operation-count estimates for the comparison protocols.

In [ ]:
WARMUP_RUNS = 100
BATCHES = 10
RUNS_PER_BATCH = 1000

for _ in range(WARMUP_RUNS):
    assert run_complete_protocol()

complete_batch_means = []
complete_batch_stds = []
complete_successes = 0
complete_failures = 0

for _ in range(BATCHES):
    batch_times = []
    for _ in range(RUNS_PER_BATCH):
        start = time.perf_counter_ns()
        result = run_complete_protocol()
        elapsed_ms = (time.perf_counter_ns() - start) / 1_000_000
        batch_times.append(elapsed_ms)
        if result:
            complete_successes += 1
        else:
            complete_failures += 1
    complete_batch_means.append(statistics.mean(batch_times))
    complete_batch_stds.append(statistics.pstdev(batch_times))

complete_all_mean = statistics.mean(complete_batch_means)
complete_all_sd = statistics.stdev(complete_batch_means) if len(complete_batch_means) > 1 else 0

complete_summary = pd.DataFrame([{
    "Warm-up runs": WARMUP_RUNS,
    "Batches": BATCHES,
    "Runs per batch": RUNS_PER_BATCH,
    "Measured executions": BATCHES * RUNS_PER_BATCH,
    "Successful executions": complete_successes,
    "Failed executions": complete_failures,
    "Mean of batch means (ms)": complete_all_mean,
    "SD of batch means (ms)": complete_all_sd
}])
display(complete_summary)

## 5. Cryptographic primitive benchmarks

Each primitive is measured in repeated batches after warm-up. Results are means of per-operation times. The ECC benchmark uses P-256 generator scalar multiplication as an explicitly identified ECC-operation proxy. Confirm that this operation corresponds to the ECC operation counted in the cited comparison protocols before presenting the derived figures as directly comparable.

In [ ]:
def benchmark_primitive(fn, warmups=1000, batches=10, iterations=10000):
    for _ in range(warmups):
        fn()
    batch_means = []
    for _ in range(batches):
        start = time.perf_counter_ns()
        for _ in range(iterations):
            fn()
        elapsed = time.perf_counter_ns() - start
        batch_means.append((elapsed / iterations) / 1_000_000)
    return {
        "mean_ms": statistics.mean(batch_means),
        "sd_ms": statistics.stdev(batch_means) if len(batch_means) > 1 else 0.0,
        "batch_means_ms": batch_means
    }

key = secrets.token_bytes(32)
payload = b"LHBAP benchmark payload"
aes_payload = b"0123456789ABCDEF"

def op_hash():
    hashlib.sha256(payload).digest()

def op_hmac():
    hmac.new(key, payload, hashlib.sha256).digest()

# AES-128 ECB encrypt + decrypt, retaining the thesis's AES encrypt/decrypt operation basis.
from Crypto.Cipher import AES
aes_key = secrets.token_bytes(16)
aes_block = b"0123456789ABCDEF"
def op_aes_encrypt_decrypt():
    cipher = AES.new(aes_key, AES.MODE_ECB)
    ciphertext = cipher.encrypt(aes_block)
    AES.new(aes_key, AES.MODE_ECB).decrypt(ciphertext)

# P-256 scalar multiplication used as a clearly documented ECC primitive proxy.
from ecdsa import NIST256p
ecc_scalar = 0x123456789ABCDEF123456789ABCDEF123456789ABCDEF123456789ABCDEF
def op_ecc_scalar_multiplication():
    NIST256p.generator * ecc_scalar

primitive_functions = {
    "TH (SHA-256)": op_hash,
    "THMAC (HMAC-SHA256)": op_hmac,
    "TEC (P-256 scalar multiplication proxy)": op_ecc_scalar_multiplication,
    "TAES (AES encrypt + decrypt)": op_aes_encrypt_decrypt
}

primitive_results = {}
for name, fn in primitive_functions.items():
    primitive_results[name] = benchmark_primitive(fn)

primitive_table = pd.DataFrame([
    {"Primitive": name,
     "Mean time (ms)": result["mean_ms"],
     "Standard deviation (ms)": result["sd_ms"]}
    for name, result in primitive_results.items()
])
display(primitive_table)

## 6. Comparative computational-cost analysis using the manuscript equations

The formulae and operation counts below are retained exactly as stated in the thesis. The measured primitive costs are substituted into those equations.

**PUF note:** a software PUF implementation or physical PUF device is not available in this notebook. Therefore, the Mostafa et al. result is reported as a measured-primitive subtotal excluding the two PUF evaluations, plus an explicitly marked optional zero-cost assumption column for comparison with the earlier manuscript table. A zero value is not a measured PUF execution time.

In [ ]:
TH = primitive_results["TH (SHA-256)"]["mean_ms"]
THMAC = primitive_results["THMAC (HMAC-SHA256)"]["mean_ms"]
TEC = primitive_results["TEC (P-256 scalar multiplication proxy)"]["mean_ms"]
TAES = primitive_results["TAES (AES encrypt + decrypt)"]["mean_ms"]

# Keep the manuscript equations and operation counts unchanged.
costs = {
    "Proposed LHBAP": 6 * THMAC,
    "Mostafa et al. (2020), excluding PUF": 8 * THMAC + 2 * TAES,
    "Wang et al. (2025) / EBIAS": 6 * TEC + 8 * TH,
    "He et al. (2023)": 10 * TEC + 13 * TH
}

# Explicitly reproduce the former manuscript's TPUF = 0 assumption, without calling it measured.
TPUF_ASSUMED_ZERO = 0.0
costs_with_puf_zero_assumption = dict(costs)
costs_with_puf_zero_assumption["Mostafa et al. (2020), TPUF=0 assumption"] = (
    8 * THMAC + 2 * TAES + 2 * TPUF_ASSUMED_ZERO
)

computational_table = pd.DataFrame([
    {"Protocol": name, "Estimated computational cost (ms)": value}
    for name, value in costs.items()
])
display(computational_table)

print("Primitive inputs (ms):")
print({"TH": TH, "THMAC": THMAC, "TEC proxy": TEC, "TAES": TAES})
print("\nMostafa et al. with the earlier TPUF=0 assumption (not measured):",
      costs_with_puf_zero_assumption["Mostafa et al. (2020), TPUF=0 assumption"], "ms")

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(computational_table["Protocol"], computational_table["Estimated computational cost (ms)"])
plt.ylabel("Estimated computational cost (ms)")
plt.xlabel("Authentication protocol")
plt.title("Operation-count-based computational cost comparison")
plt.xticks(rotation=15, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Comparative communication-cost analysis

The message fields and bit lengths below are transcribed from the thesis. The notebook calculates each message size, each direction's aggregate, and the total communication cost. These are analytical message-size calculations, not packet captures or measurements of radio-layer framing overhead.

In [ ]:
# Field sizes (bits) and messages as specified in the thesis.
communication_specs = {
    "Proposed LHBAP": {
        "Sensor -> Gateway": [
            ("ID_SN", 160), ("Ns", 128), ("Ts", 32), ("Auth1", 256)
        ],
        "Gateway -> Sensor": [
            ("ID_GW", 160), ("Ng", 128), ("Tg", 32), ("Auth2", 256)
        ]
    },
    "Mostafa et al. (2020)": {
        "Sensor -> Gateway": [
            ("M1: ID_S", 160), ("M1: TS1", 32), ("M1: HMAC", 256),
            ("M3: TS3", 32), ("M3: HMAC", 256)
        ],
        "Gateway -> Sensor": [
            ("M2: C", 128), ("M2: TS2", 32), ("M2: HMAC", 256),
            ("M4: M", 128), ("M4: TS4", 32), ("M4: HMAC", 256)
        ]
    },
    "He et al. (2023)": {
        "Sensor -> Gateway": [
            ("PIDi", 160), ("P1", 320), ("Y", 256), ("T1", 32)
        ],
        "Gateway -> Sensor": [
            ("P3", 320), ("P4", 320), ("Z", 256), ("T3", 32)
        ]
    },
    "Wang et al. (2025) / EBIAS": {
        "Sensor -> Gateway": [
            ("PID", 160), ("P1", 320), ("P2", 256), ("Y", 256), ("T", 256)
        ],
        "Gateway -> Sensor": [
            ("P3", 320), ("P4", 320), ("Z", 256)
        ]
    }
}

communication_rows = []
for protocol, directions in communication_specs.items():
    direction_totals = {}
    for direction, fields in directions.items():
        total = sum(bits for _, bits in fields)
        direction_totals[direction] = total
        communication_rows.append({
            "Protocol": protocol,
            "Direction": direction,
            "Fields": ", ".join(f"{name}={bits}" for name,bits in fields),
            "Communication cost (bits)": total
        })
    communication_rows.append({
        "Protocol": protocol,
        "Direction": "Total",
        "Fields": "",
        "Communication cost (bits)": sum(direction_totals.values())
    })

communication_detail = pd.DataFrame(communication_rows)
display(communication_detail)

In [ ]:
communication_totals = communication_detail[communication_detail["Direction"] == "Total"].copy()
communication_totals = communication_totals[["Protocol", "Communication cost (bits)"]]
display(communication_totals)

plt.figure(figsize=(9, 5))
plt.bar(communication_totals["Protocol"], communication_totals["Communication cost (bits)"])
plt.ylabel("Communication cost (bits)")
plt.xlabel("Authentication protocol")
plt.title("Communication cost comparison")
plt.xticks(rotation=15, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 8. Consolidated comparison tables

This section combines the operation-count-based computational estimates and analytical communication costs. Keep the distinction between measured primitive estimates and complete-protocol execution time when interpreting the results.

In [ ]:
combined = computational_table.merge(communication_totals, on="Protocol", how="outer")
display(combined)

## 9. Export results

The following cell saves the generated tables as CSV files in the Colab working directory. Download them from the Files panel if needed for thesis tables.

In [ ]:
primitive_table.to_csv("LHBAP_primitive_benchmark_results.csv", index=False)
computational_table.to_csv("LHBAP_computational_cost_comparison.csv", index=False)
communication_detail.to_csv("LHBAP_communication_cost_details.csv", index=False)
communication_totals.to_csv("LHBAP_communication_cost_comparison.csv", index=False)
combined.to_csv("LHBAP_integrated_comparison.csv", index=False)
print("CSV result files saved.")

## 10. Reporting and interpretation notes

1. Report the Colab runtime details printed in Section 1 for the actual run used in the thesis revision.
2. Report the number of warm-up executions, batches, and measured iterations exactly as shown in the notebook outputs.
3. The complete LHBAP runtime is a full software-simulation measurement. The comparative protocol values are operation-count-based estimates using measured primitive costs.
4. The ECC value uses P-256 scalar multiplication as a stated proxy. Before treating it as equivalent to \(T_{EC}\) in the cited papers, verify the exact ECC operation counted in those papers. If the papers define a different operation, benchmark that operation instead.
5. PUF timing is not measured. Do not report \(T_{PUF}=0\) as a measured result. The zero-cost column only reproduces the earlier simplifying assumption.
6. Communication totals use the thesis's field sizes. They exclude lower-layer headers, retransmissions, framing, and other network overhead unless these are explicitly included in the message specification.
7. Do not describe the comparison protocols as fully implemented unless their complete algorithms and message flows have actually been implemented from the cited publications.
8. The nonce replay tests validate the behaviour of this in-memory simulation. A deployed system requires replay-state persistence and lifecycle management.